# DAD (Dynamic Affinity Dock) — Colab Reference Runner

**Stages 0–12 — sequential cell execution**

This notebook is the primary execution path for the DAD pipeline.
Snakemake is used for workflow definition and dry-run validation;
actual GPU computation (ColabFold, GNINA) runs here.

**Worked example inputs (AW1_ref / Tier 1):**
- Proteins: MCP (NA23_RS01195), Crp (NA23_RS08105), RbsB (NA23_RS00870)
- Ligands: Ala-Ile, Gly-Val (dipeptides.smi)
- Existing structures: `AW1_ref/structure_{MCP,Crp,Rbs}.pdb`
- Existing pockets:   `AW1_ref/structure.pdb_predictions_{MCP,Crp,Rbs}.csv`

**Ground truth (Revision.txt):**

| Target | Ligand | Vina | CNN pose | CNN affinity |
|--------|--------|------|----------|--------------|
| MCP    | Ala-Ile | -5.61 | 0.8995 | 4.416 |
| MCP    | Gly-Val | -5.18 | 0.8468 | 4.022 |
| Crp    | Ala-Ile | -6.01 | 0.6228 | 4.642 |
| Crp    | Gly-Val | -5.59 | 0.7353 | 4.574 |
| RbsB   | Ala-Ile | -4.92 | 0.4447 | 3.516 |
| RbsB   | Gly-Val | -4.44 | 0.9144 | 3.896 |


## Stage 0 — Environment setup

In [ ]:
# Install dependencies (run once per Colab session)
import subprocess, sys

PACKAGES = [
    'colabfold[alphafold-minus-jax]',
    'biopython',
    'rdkit-pypi',
    'plip',
    'requests',
    'numpy',
    'pandas',
]
for pkg in PACKAGES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

# Install GNINA binary (Linux only)
import os
if not os.path.exists('/usr/local/bin/gnina'):
    subprocess.run([
        'wget', '-q', '-O', '/usr/local/bin/gnina',
        'https://github.com/gnina/gnina/releases/download/v1.3.2/gnina'
    ], check=True)
    os.chmod('/usr/local/bin/gnina', 0o755)
    print('GNINA 1.3.2 installed.')
else:
    print('GNINA already present.')

print('Environment ready.')

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys
from pathlib import Path

# --- Configure paths below ---
DRIVE_ROOT   = '/content/drive/MyDrive'          # adjust if different
DAD_ROOT     = f'{DRIVE_ROOT}/DAD'               # where this repo lives on Drive
PIPELINE_DIR = f'{DAD_ROOT}/06_Report/Mr_Pipeline'
AW1_REF      = f'{DAD_ROOT}/AW1_ref'
TIER1_INPUT  = f'{DAD_ROOT}/01_Tier1_input'
RESULTS      = f'{PIPELINE_DIR}/results'

# Add dad package to Python path
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

Path(RESULTS).mkdir(parents=True, exist_ok=True)
print(f'DAD_ROOT: {DAD_ROOT}')
print(f'AW1_REF:  {AW1_REF}')

## Stage 1 — Input ingestion

In [ ]:
import json
from pathlib import Path
from dad.io import ProteinInput, LigandInput
from Bio import SeqIO

FASTA_PATH  = f'{TIER1_INPUT}/proteins'  # folder with 3 individual FASTA files
SMILES_PATH = f'{TIER1_INPUT}/ligands/dipeptides.smi'

# Load proteins from all FASTA files in the proteins folder
proteins = []
for fasta_file in sorted(Path(FASTA_PATH).glob('*.fasta')):
    for rec in SeqIO.parse(str(fasta_file), 'fasta'):
        proteins.append(ProteinInput(
            seq_id=str(rec.id),
            sequence=str(rec.seq),
            source_file=str(fasta_file.resolve()),
            description=str(rec.description),
            organism='Fervidobacterium islandicum AW-1',
        ))

# Load ligands from SMILES file (format: SMILES<tab>ID)
ligands = []
for line in open(SMILES_PATH):
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    parts = line.split()
    smiles, lig_id = (parts[0], parts[1]) if len(parts) >= 2 else (parts[0], f'lig_{len(ligands)+1}')
    ligands.append(LigandInput(lig_id=lig_id, smiles=smiles, source_file=SMILES_PATH))

# Write JSONL
ingest_dir = Path(RESULTS) / 'ingest'
ingest_dir.mkdir(parents=True, exist_ok=True)

with open(ingest_dir / 'proteins.jsonl', 'w') as f:
    for p in proteins: f.write(json.dumps(p.to_dict()) + '\n')

with open(ingest_dir / 'ligands.jsonl', 'w') as f:
    for l in ligands: f.write(json.dumps(l.to_dict()) + '\n')

# Ligand index for Snakemake
lig_idx = Path(RESULTS) / 'ligands' / 'index.txt'
lig_idx.parent.mkdir(parents=True, exist_ok=True)
lig_idx.write_text('\n'.join(l.lig_id for l in ligands) + '\n')

print(f'Ingested {len(proteins)} proteins: {[p.seq_id for p in proteins]}')
print(f'Ingested {len(ligands)} ligands: {[l.lig_id for l in ligands]}')

## Stage 2 — Sequence QC & dereplication

In [ ]:
# Length filter (MMseqs2 clustering skipped for Tier 1; all 3 proteins are unique)
MIN_LEN, MAX_LEN = 30, 5000

qc_dir = Path(RESULTS) / 'qc'
qc_dir.mkdir(parents=True, exist_ok=True)

proteins_qc = [p for p in proteins if MIN_LEN <= len(p.sequence) <= MAX_LEN]

with open(qc_dir / 'proteins_qc.jsonl', 'w') as f:
    for p in proteins_qc: f.write(json.dumps(p.to_dict()) + '\n')

with open(qc_dir / 'clusters.tsv', 'w') as f:
    f.write('representative\tmember\n')
    for p in proteins_qc: f.write(f'{p.seq_id}\t{p.seq_id}\n')

print(f'QC: {len(proteins)} → {len(proteins_qc)} proteins retained.')

## Stage 3 — Pre-docking biological triage

In [ ]:
from dad.core.triage import run_triage, TriageConfig, write_triage_report, write_pass_fasta

triage_dir = Path(RESULTS) / 'triage'
triage_dir.mkdir(parents=True, exist_ok=True)
(triage_dir / 'topology').mkdir(exist_ok=True)
(triage_dir / 'annotation').mkdir(exist_ok=True)

sequences = {p.seq_id: p.sequence for p in proteins_qc}

# config.yaml triage_tool: 'deeptmhmm' (default) or 'phobius'
triage_cfg = TriageConfig(triage_tool='DeepTMHMM')

records = run_triage(
    sequences=sequences,
    config=triage_cfg,
    source_tier='Tier1',
    output_tsv=triage_dir / 'triage_report.tsv',
    output_fasta=triage_dir / 'triage_pass.fasta',
)

# Inclusion policy (rationale.md §R5):
#   PASS / PASS_CLIPPED -> proceed normally
#   FLAG                -> proceed with warning (low-confidence annotation, default: include)
#   EXCLUDE             -> dropped
accepted = [r for r in records if r.status.value in ('PASS', 'PASS_CLIPPED', 'FLAG')]
flagged  = [r for r in records if r.status.value == 'FLAG']

print(f'Triage: {len(accepted)}/{len(records)} ORFs accepted for structure prediction.')
if flagged:
    print(f'WARNING: {len(flagged)} FLAG ORF(s) included with low-confidence annotation:')
    for r in flagged:
        print(f'  [FLAG] {r.orf_id}: {r.notes}')
for r in records:
    print(f'  {r.orf_id}: {r.status.value} | nTM={r.n_tm} | dock=[{r.dock_region_start}:{r.dock_region_end}] | {r.functional_class.value}')

## Stage 4 — Structure prediction (AW1_ref reuse)

In [ ]:
from dad.core.structure import (
    load_existing_pdb, load_af3_results, run_colabfold,
    assess_structure_quality, select_dock_region_pdb,
)
from dad.io import StructurePrediction
import shutil

struct_dir = Path(RESULTS) / 'structures'
PLDDT_MIN = 70.0

# AW1_ref PDB map (seq_id substring -> file)
AW1_PDB_MAP = {
    'NA23_RS01195': f'{AW1_REF}/structure_MCP.pdb',
    'NA23_RS08105': f'{AW1_REF}/structure_Crp.pdb',
    'NA23_RS00870': f'{AW1_REF}/structure_Rbs.pdb',
}

# Triage record map for dock_region coords
triage_map = {r.orf_id: r for r in records}

structures = []  # List[StructurePrediction]
for p in proteins_qc:
    rec = triage_map.get(p.seq_id)
    if rec is None or rec.status.value == 'EXCLUDE':
        continue

    job_dir = struct_dir / p.seq_id
    job_dir.mkdir(parents=True, exist_ok=True)

    rank1_pdb = str(job_dir / 'rank_1.pdb')
    dock_region_pdb = str(job_dir / 'dock_region.pdb')

    # Reuse AW1_ref if available
    aw1_pdb = None
    for key, fpath in AW1_PDB_MAP.items():
        if key in p.seq_id and Path(fpath).exists():
            aw1_pdb = fpath
            break

    if aw1_pdb:
        shutil.copy2(aw1_pdb, rank1_pdb)
        sp = load_existing_pdb(rank1_pdb, p.seq_id, plddt_min=PLDDT_MIN)
        print(f'[AW1_ref] {p.seq_id}: pLDDT={sp.mean_plddt:.1f}')
    else:
        # ColabFold (requires GPU)
        print(f'[ColabFold] Running for {p.seq_id}...')
        seq = rec.dock_seq if rec.dock_seq else p.sequence
        run_colabfold(seq_id=p.seq_id, sequence=seq, job_dir=str(job_dir))
        sp = load_existing_pdb(rank1_pdb, p.seq_id, plddt_min=PLDDT_MIN)

    # Write quality.json
    quality = assess_structure_quality(rank1_pdb, plddt_min=PLDDT_MIN)
    import json as _json
    (job_dir / 'quality.json').write_text(_json.dumps({**quality, 'seq_id': p.seq_id}, indent=2))

    # Extract dock_region
    dock_start = rec.dock_region_start if rec.dock_region_start else None
    dock_end   = rec.dock_region_end   if rec.dock_region_end   else None
    select_dock_region_pdb(
        pdb_path=rank1_pdb,
        dock_region_start=dock_start,
        dock_region_end=dock_end,
        output_path=dock_region_pdb,
    )
    structures.append(sp)

print(f'\nStructures ready: {[s.seq_id for s in structures]}')

## Stage 5 — Pocket detection (P2Rank / AW1_ref reuse)

In [ ]:
from dad.core.pocket import detect_pockets, load_existing_predictions_csv
from dad.io import PocketResult

pocket_dir = Path(RESULTS) / 'pockets'
TOP_N = 3

# AW1_ref pocket CSV map
AW1_CSV_MAP = {
    'NA23_RS01195': f'{AW1_REF}/structure.pdb_predictions_MCP.csv',
    'NA23_RS08105': f'{AW1_REF}/structure.pdb_predictions_Crp.csv',
    'NA23_RS00870': f'{AW1_REF}/structure.pdb_predictions_Rbs.csv',
}

pocket_results = {}  # Dict[seq_id, List[PocketResult]]

for sp in structures:
    job_dir = pocket_dir / sp.seq_id
    job_dir.mkdir(parents=True, exist_ok=True)

    aw1_csv = None
    for key, fpath in AW1_CSV_MAP.items():
        if key in sp.seq_id and Path(fpath).exists():
            aw1_csv = fpath
            break

    if aw1_csv:
        pockets = load_existing_predictions_csv(aw1_csv, sp.seq_id, top_n=TOP_N)
        shutil.copy2(aw1_csv, job_dir / 'predictions.csv')
        print(f'[AW1_ref] {sp.seq_id}: {len(pockets)} pockets loaded.')
    else:
        dock_region = str(struct_dir / sp.seq_id / 'dock_region.pdb')
        from dad.core.pocket import run_p2rank, parse_p2rank_output
        pred_csv = run_p2rank(dock_region, str(job_dir), p2rank_path='prank', alphafold_mode=True)
        pockets = parse_p2rank_output(pred_csv, sp.seq_id, top_n=TOP_N)
        print(f'[P2Rank] {sp.seq_id}: {len(pockets)} pockets detected.')

    # Write pockets.jsonl
    with open(job_dir / 'pockets.jsonl', 'w') as f:
        for pk in pockets: f.write(json.dumps(pk.to_dict()) + '\n')

    pocket_results[sp.seq_id] = pockets
    for pk in pockets:
        print(f'  pocket{pk.pocket_rank}: score={pk.score:.2f}, center=({pk.center_x:.2f},{pk.center_y:.2f},{pk.center_z:.2f})')

## Stage 6 — Ligand preparation

In [ ]:
from dad.core.docking import smiles_to_3d_sdf
from dad.io import PreparedLigand

lig_dir = Path(RESULTS) / 'ligands'
prepared_ligands = {}  # Dict[lig_id, PreparedLigand]

for lig in ligands:
    out_dir = lig_dir / lig.lig_id
    out_dir.mkdir(parents=True, exist_ok=True)
    sdf_path = str(out_dir / 'prepared.sdf')

    prep = smiles_to_3d_sdf(
        smiles=lig.smiles,
        lig_id=lig.lig_id,
        output_sdf=sdf_path,
        ph=7.4,
    )
    (out_dir / 'meta.json').write_text(json.dumps(prep.to_dict(), indent=2))
    prepared_ligands[lig.lig_id] = prep
    print(f'{lig.lig_id}: MW={prep.mol_weight:.1f} Da, max_dim={prep.max_dim:.2f} Å → {sdf_path}')

## Stage 7 — Auto box configuration

In [ ]:
from dad.core.pocket import pocket_to_box
from dad.io import DockingBox

dock_dir = Path(RESULTS) / 'docking'
BOX_SIZE_MIN, BOX_PADDING = 22.0, 10.0

docking_boxes = []  # List[DockingBox]

for seq_id, pockets in pocket_results.items():
    for pk in pockets:
        for lig_id, prep in prepared_ligands.items():
            box_dict = pocket_to_box(pk, prep.max_dim, BOX_SIZE_MIN, BOX_PADDING)
            box = DockingBox(
                seq_id=seq_id, lig_id=lig_id, pocket_rank=pk.pocket_rank,
                **box_dict,
            )
            # Write box.json
            box_out = dock_dir / seq_id / lig_id / f'pocket{pk.pocket_rank}'
            box_out.mkdir(parents=True, exist_ok=True)
            (box_out / 'box.json').write_text(json.dumps(box.to_dict(), indent=2))
            docking_boxes.append(box)

print(f'Generated {len(docking_boxes)} docking boxes ({len(pocket_results)} proteins × {len(prepared_ligands)} ligands × up to {TOP_N} pockets).')

## Stage 8 — GNINA docking

In [ ]:
from dad.core.docking import run_gnina_single
from dad.io import DockingResult

GNINA_PATH = '/usr/local/bin/gnina'  # installed in Stage 0

docking_results = []  # List[DockingResult]

for box in docking_boxes:
    receptor = str(struct_dir / box.seq_id / 'dock_region.pdb')
    ligand   = str(lig_dir / box.lig_id / 'prepared.sdf')
    out_sdf  = str(dock_dir / box.seq_id / box.lig_id / f'pocket{box.pocket_rank}' / 'docked.sdf')
    scores_json = str(dock_dir / box.seq_id / box.lig_id / f'pocket{box.pocket_rank}' / 'scores.json')

    result = run_gnina_single(
        receptor_pdb=receptor,
        ligand_sdf=ligand,
        box=box,
        output_sdf=out_sdf,
        gnina_path=GNINA_PATH,
        exhaustiveness=32,
        num_modes=9,
        seed=0,
    )
    Path(scores_json).write_text(json.dumps(result.to_dict(), indent=2))
    docking_results.append(result)

    bp = result.best_pose
    print(f'{box.seq_id} × {box.lig_id} pocket{box.pocket_rank}: '
          f'vina={bp.vina_affinity:.3f}  cnn_pose={bp.cnn_pose_score:.4f}  cnn_aff={bp.cnn_affinity:.3f}  ({result.elapsed_seconds:.0f}s)')

## Stage 8b — Validation against ground truth (Revision.txt)

In [ ]:
# Ground truth from Revision.txt
GROUND_TRUTH = {
    ('NA23_RS01195', 'Ala-Ile'): {'vina': -5.61, 'cnn_pose': 0.8995, 'cnn_aff': 4.416},
    ('NA23_RS01195', 'Gly-Val'): {'vina': -5.18, 'cnn_pose': 0.8468, 'cnn_aff': 4.022},
    ('NA23_RS08105', 'Ala-Ile'): {'vina': -6.01, 'cnn_pose': 0.6228, 'cnn_aff': 4.642},
    ('NA23_RS08105', 'Gly-Val'): {'vina': -5.59, 'cnn_pose': 0.7353, 'cnn_aff': 4.574},
    ('NA23_RS00870', 'Ala-Ile'): {'vina': -4.92, 'cnn_pose': 0.4447, 'cnn_aff': 3.516},
    ('NA23_RS00870', 'Gly-Val'): {'vina': -4.44, 'cnn_pose': 0.9144, 'cnn_aff': 3.896},
}

print(f'{"Target":<25} {"Ligand":<10} {"Vina":<8} {"GT_Vina":<10} {"CNN_pose":<10} {"GT_pose":<10} {"CNN_aff":<10} {"GT_aff"}')
print('-' * 100)
for res in docking_results:
    key = (res.seq_id, res.lig_id)
    if key not in GROUND_TRUTH:
        continue
    gt = GROUND_TRUTH[key]
    bp = res.best_pose
    if res.pocket_rank == 1:  # show only top pocket
        print(f'{res.seq_id:<25} {res.lig_id:<10} '
              f'{bp.vina_affinity:<8.3f} {gt["vina"]:<10.3f} '
              f'{bp.cnn_pose_score:<10.4f} {gt["cnn_pose"]:<10.4f} '
              f'{bp.cnn_affinity:<10.3f} {gt["cnn_aff"]:.3f}')

## Stage 9 — Interaction profiling

In [ ]:
from dad.core.interaction import extract_contact_residues, run_plip
from dad.io import InteractionProfile

inter_dir = Path(RESULTS) / 'interactions'
CONTACT_DIST = 5.0
USE_PLIP = True

interaction_profiles = []  # List[InteractionProfile]

for res in docking_results:
    receptor  = str(struct_dir / res.seq_id / 'dock_region.pdb')
    docked    = res.output_sdf
    out_base  = inter_dir / res.seq_id / res.lig_id / f'pocket{res.pocket_rank}'
    out_base.mkdir(parents=True, exist_ok=True)

    contacts = extract_contact_residues(receptor, docked, pose_index=0, contact_distance=CONTACT_DIST)

    with open(out_base / 'contacts.jsonl', 'w') as f:
        for c in contacts: f.write(json.dumps(c.to_dict()) + '\n')

    n_hbonds = n_hydrophobic = None
    plip_xml_path = None
    if USE_PLIP:
        try:
            plip_result = run_plip(receptor, docked, str(out_base / 'plip.xml'))
            n_hbonds = plip_result.get('n_hbonds')
            n_hydrophobic = plip_result.get('n_hydrophobic')
            plip_xml_path = str(out_base / 'plip.xml')
        except Exception as e:
            print(f'PLIP skipped for {res.seq_id}×{res.lig_id}: {e}')

    profile = InteractionProfile(
        seq_id=res.seq_id, lig_id=res.lig_id,
        pocket_rank=res.pocket_rank, pose_rank=1,
        contact_residues=contacts,
        n_hbonds=n_hbonds, n_hydrophobic=n_hydrophobic,
        plip_xml_path=plip_xml_path,
    )
    (out_base / 'profile.json').write_text(json.dumps(profile.to_dict(), indent=2))
    interaction_profiles.append(profile)
    print(f'{res.seq_id} × {res.lig_id} pocket{res.pocket_rank}: {len(contacts)} contacts, hbonds={n_hbonds}')

## Stage 10 — Aggregation & ranking

In [ ]:
from dad.core.interaction import aggregate_results, write_master_csv, build_heatmap_matrix

report_dir = Path(RESULTS) / 'report'
report_dir.mkdir(parents=True, exist_ok=True)

weights = {'vina': 1.0, 'cnn_pose': 1.0, 'cnn_affinity': 1.0}
ranked = aggregate_results(docking_results, interaction_profiles, weights)

master_csv = str(report_dir / 'docking_master.csv')
write_master_csv(ranked, master_csv)

print(f'Master CSV written: {master_csv}')
print(f'\nTop 5 ranked pairs:')
for r in ranked[:5]:
    print(f'  rank={r.overall_rank}: {r.seq_id} × {r.lig_id} pocket{r.pocket_rank} '
          f'vina={r.vina_affinity:.3f} composite={r.composite_rank_score:.4f}')

## Stage 11 — Visualization & HTML report

In [ ]:
from dad.core.visualize import run_visualization_batch, generate_html_report

viz_dir = Path(RESULTS) / 'visualization'
viz_dir.mkdir(parents=True, exist_ok=True)

viz_inputs = []
for res in docking_results:
    receptor = str(struct_dir / res.seq_id / 'dock_region.pdb')
    profile_path = inter_dir / res.seq_id / res.lig_id / f'pocket{res.pocket_rank}' / 'profile.json'
    profile = None
    if profile_path.exists():
        profile = InteractionProfile.from_dict(json.loads(profile_path.read_text()))

    viz_inputs.append({
        'receptor_pdb': receptor,
        'docked_sdf': res.output_sdf,
        'profile': profile,
        'seq_id': res.seq_id,
        'lig_id': res.lig_id,
        'pocket': f'pocket{res.pocket_rank}',
        'output_dir': str(viz_dir),
        'width': 900, 'height': 600,
        'generate_cxc': True,
    })

run_visualization_batch(viz_inputs)
report_html = str(report_dir / 'dad_report.html')
generate_html_report(master_csv=master_csv, viz_dir=str(viz_dir), output_html=report_html, top_k=10)

print(f'Report: {report_html}')
# Display inline in Colab
from IPython.display import IFrame
IFrame(src=report_html, width='100%', height=600)

## Stage 12 — Reproducibility manifest

In [ ]:
import hashlib, datetime, subprocess as _sp

def _ver(cmd):
    try:
        r = _sp.run(cmd, capture_output=True, text=True, timeout=5)
        return r.stdout.strip().splitlines()[0] if r.returncode == 0 else 'unavailable'
    except Exception:
        return 'unavailable'

def _sha256(path):
    h = hashlib.sha256()
    try:
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(65536), b''): h.update(chunk)
        return h.hexdigest()
    except Exception:
        return 'unavailable'

try:
    import rdkit; rdkit_ver = rdkit.__version__
except: rdkit_ver = 'unavailable'
try:
    import Bio; bio_ver = Bio.__version__
except: bio_ver = 'unavailable'

manifest = {
    'dad_version': '0.1.0-beta',
    'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'execution_env': 'Google Colab',
    'tool_versions': {
        'gnina': _ver([GNINA_PATH, '--version']),
        'rdkit': rdkit_ver,
        'biopython': bio_ver,
        'plip': _ver(['plip', '--version']),
        'colabfold': _ver(['colabfold_batch', '--version']),
    },
    'output_checksums': {
        'docking_master.csv': _sha256(master_csv),
        'dad_report.html': _sha256(report_html),
    },
    'parameters': {
        'exhaustiveness': 32, 'num_modes': 9, 'seed': 0,
        'box_size_min': BOX_SIZE_MIN, 'box_padding': BOX_PADDING,
        'contact_distance': CONTACT_DIST, 'plddt_min': PLDDT_MIN,
    },
}

manifest_path = str(Path(RESULTS) / 'manifest.json')
Path(manifest_path).write_text(json.dumps(manifest, indent=2))
print(f'Manifest written: {manifest_path}')
print(json.dumps(manifest, indent=2))